In [1]:
import subprocess, sys

subprocess.check_call([sys.executable, "-m", "pip", "install",
    "deepface", "tensorflow", "tf-keras", "pytubefix",
    "opencv-python", "Pillow", "numpy", "pandas", "openpyxl"])

0

In [2]:
import subprocess
import numpy as np
from PIL import Image
import io
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from deepface import DeepFace
from typing import List

NUM_FRAMES = 50
MAX_WORKERS = 1
LOG_FILE = "completed.txt"
RESULTS_FILE = "emotion_results.csv"
INPUT_FILE = "Player Stats/nba_postgame_interviews_with_gmsc.xlsx"

In [3]:
def get_stream_url(url):
    from pytubefix import YouTube
    import time
    
    for attempt in range(3):  # retry up to 3 times
        try:
            yt = YouTube(url, use_oauth=True, allow_oauth_cache=True)
            stream = yt.streams.filter(progressive=True, file_extension='mp4').order_by('resolution').first()
            if stream is None:
                raise RuntimeError("No suitable stream found")
            return stream.url
        except Exception as e:
            if attempt < 2:
                print(f"  Retry {attempt+1}/3 for {url}")
                time.sleep(3)
            else:
                raise e

def get_video_duration(stream_url):
    cmd = ['ffprobe', '-v', 'error',
           '-show_entries', 'format=duration',
           '-of', 'default=noprint_wrappers=1:nokey=1',
           stream_url]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return float(result.stdout.strip())

def extract_frame_at(stream_url, timestamp):
    cmd = ['ffmpeg', '-ss', str(timestamp), '-i', stream_url,
           '-frames:v', '1', '-f', 'image2pipe',
           '-pix_fmt', 'rgb24', '-vf', 'scale=640:480',
           '-loglevel', 'error', 'pipe:']
    result = subprocess.run(cmd, capture_output=True)
    try:
        return np.array(Image.open(io.BytesIO(result.stdout)))
    except Exception:
        return None

def analyze_frame(frame, frame_idx, timestamp):
    try:
        analysis = DeepFace.analyze(img_path=frame, actions=['emotion'],
                                    enforce_detection=False, silent=True)
        face = analysis[0]
        return {
            'frame': frame_idx,
            'timestamp': round(timestamp, 2),
            'face_confidence': round(face.get('face_confidence', 0), 4),
            'dominant_emotion': face['dominant_emotion'],
            **{k: round(float(v), 4) for k, v in face['emotion'].items()}
        }
    except Exception as e:
        print(f"  [!] Frame {frame_idx} failed: {e}")
        return None

def get_video_id(url):
    if 'youtube.com' in url:
        return url.split('v=')[-1].split('&')[0]
    return url.split('/')[-1].split('?')[0]

def sanitize(name):
    return "".join(c for c in name if c not in r'\/:*?"<>|').strip()

def already_completed(video_id):
    if not os.path.exists(LOG_FILE):
        return False
    with open(LOG_FILE) as f:
        return video_id in f.read()

def mark_completed(video_id):
    with open(LOG_FILE, 'a') as f:
        f.write(video_id + '\n')

def average_emotions(frame_results):
    emotion_cols = ['angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral']
    averaged = {}
    for emotion in emotion_cols:
        values = [r[emotion] for r in frame_results if emotion in r]
        averaged[f'avg_{emotion}'] = round(np.mean(values), 4) if values else None
    dominant_counts = pd.Series([r['dominant_emotion'] for r in frame_results])
    averaged['dominant_emotion'] = dominant_counts.mode()[0] if not dominant_counts.empty else None
    averaged['frames_analyzed'] = len(frame_results)
    return averaged

In [4]:
def process_video(row):
    url = row['link']
    player = sanitize(str(row['name']))
    date = sanitize(str(row['upload date']))
    video_id = get_video_id(url)
    label = f"{player} / {date}"

    if already_completed(video_id):
        print(f"[SKIP] {label}")
        return []

    print(f"[START] {label}")
    try:
        stream_url = get_stream_url(url)
        duration = get_video_duration(stream_url)
        timestamps = np.linspace(0, duration, NUM_FRAMES, endpoint=False)
    except Exception as e:
        print(f"[FAIL] {label}: {e}")
        return []

    results = []
    for i, ts in enumerate(timestamps):
        print(f"  Frame {i+1}/{NUM_FRAMES} (t={ts:.1f}s)...", end='\r')
        frame = extract_frame_at(stream_url, ts)
        if frame is None:
            continue
        result = analyze_frame(frame, i, ts)
        if result:
            result.update({'video_id': video_id, 'player': player,
                           'upload_date': date, 'url': url})
            results.append(result)

    print(f"  Done — {len(results)}/{NUM_FRAMES} frames analyzed      ")
    mark_completed(video_id)
    return results

In [5]:
df = pd.read_excel(INPUT_FILE)
df.columns = df.columns.str.lower()
rows = df.to_dict('records')
print(f"Loaded {len(rows)} videos")

all_results = []

# Load existing results if they exist
if os.path.exists(RESULTS_FILE):
    existing_df = pd.read_csv(RESULTS_FILE)
    all_results = existing_df.to_dict('records')
    print(f"Loaded {len(all_results)} existing results")

all_results = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_video, row): row for row in rows}
    for future in as_completed(futures):
        try:
            frame_results = future.result()
            if not frame_results:
                continue
            averaged = average_emotions(frame_results)
            first = frame_results[0]
            all_results.append({
                'player': first['player'],
                'upload_date': first['upload_date'],
                'video_id': first['video_id'],
                'url': first['url'],
                **averaged
            })
            pd.DataFrame(all_results).to_csv(RESULTS_FILE, index=False)
            print(f"[SAVED] {first['player']} / {first['upload_date']}")
        except Exception as e:
            print(f"[ERROR] {e}")

print(f"\nDone! {len(all_results)} videos saved to {RESULTS_FILE}")

Loaded 610 videos
Loaded 3 existing results
[SKIP] Andrew Wiggins / 2022-06-09 000000
[SKIP] Andrew Wiggins / 2022-06-03 000000
[SKIP] Andrew Wiggins / 2022-05-27 000000
[SKIP] Andrew Wiggins / 2022-05-23 000000
[SKIP] Andrew Wiggins / 2022-05-19 000000
[SKIP] Andrew Wiggins / 2022-05-10 000000
[SKIP] Andrew Wiggins / 2022-05-04 000000
[SKIP] Andrew Wiggins / 2022-03-24 000000
[SKIP] Andrew Wiggins / 2022-03-13 000000
[SKIP] Andrew Wiggins / 2022-03-02 000000
[SKIP] Andrew Wiggins / 2022-02-15 000000
[SKIP] Andrew Wiggins / 2022-02-10 000000
[SKIP] Andrew Wiggins / 2022-01-30 000000
[SKIP] Andrew Wiggins / 2022-01-28 000000
[SKIP] Andrew Wiggins / 2022-01-24 000000
[SKIP] Andrew Wiggins / 2022-01-19 000000
[SKIP] Andrew Wiggins / 2022-01-04 000000
[SKIP] Andrew Wiggins / 2022-01-02 000000
[SKIP] Andrew Wiggins / 2021-12-18 000000
[SKIP] Andrew Wiggins / 2021-12-15 000000
[SKIP] Andrew Wiggins / 2021-12-07 000000
[SKIP] Andrew Wiggins / 2021-12-04 000000
[SKIP] Andrew Wiggins / 2021-11-

In [13]:
results_df = pd.read_csv(RESULTS_FILE)
results_df

,player,upload_date,video_id,url,avg_angry,avg_disgust,avg_fear,avg_happy,avg_sad,avg_surprise,avg_neutral,dominant_emotion,frames_analyzed
0,Kevin Durant,2022-03-17 000000,50sZu6E8Gh0,https://www.youtube.com/watch?v=50sZu6E8Gh0,15.6869,0.0383,24.6977,0.4137,27.6885,8.0476,23.4272,sad,50
1,Kevin Durant,2022-03-16 000000,bAF0jv9l94Q,https://www.youtube.com/watch?v=bAF0jv9l94Q,4.0956,0.0082,26.4900,1.5509,25.7349,4.0074,38.1131,neutral,50
2,Kevin Durant,2022-03-11 000000,eIFn2_Dabz0,https://www.youtube.com/watch?v=eIFn2_Dabz0,5.2070,0.9398,51.1414,0.2044,14.8504,15.2126,12.4444,fear,50
3,Kevin Durant,2022-03-09 000000,jgdvcxzLTwU,https://www.youtube.com/watch?v=jgdvcxzLTwU,3.4157,0.0006,3.2370,72.1432,7.5163,0.9274,12.7597,happy,50
4,Kevin Durant,2022-03-04 000000,q62lDKjhC1M,https://www.youtube.com/watch?v=q62lDKjhC1M,22.3876,0.1962,14.0447,17.9907,36.1157,1.6584,7.6067,sad,50
...,...,...,...,...,...,...,...,...,...,...,...,...,...
155,Steph Curry,2021-11-27 000000,LavfQhWsZm0,https://www.youtube.com/watch?v=LavfQhWsZm0,8.2194,0.0256,23.5610,8.3704,30.7485,0.1147,28.9603,sad,50
156,Steph Curry,2021-11-25 000000,JEOfpG9hqVY,https://www.youtube.com/watch?v=JEOfpG9hqVY,2.7178,0.0011,2.3872,72.0015,8.9606,1.8559,12.0759,happy,50
157,Steph Curry,2021-10-29 000000,TBIsOS9tXZo,https://www.youtube.com/watch?v=TBIsOS9tXZo,16.2062,0.3483,13.4691,0.7578,45.8963,0.4124,22.9099,sad,50
158,Terance Mann,2022-03-07 000000,njfUKflbL0s,https://www.youtube.com/watch?v=njfUKflbL0s,67.9563,0.0025,13.1948,0.4970,2.8557,0.4329,15.0609,angry,50
